# Dataset YOLO UA-DETRAC (car + van) prêt — frame redundancy réduite.

In [ ]:
import xml.etree.ElementTree as ET
import shutil
from glob import glob
import os
import cv2

# ================================================
# CONFIGURATION
# Classe finale
FINAL_CLASSES = ["car"]

# UA-DETRAC types autorisés
KEEP_TYPES = ["car", "van"]   # fusionnés
IGNORE_TYPES = ["bus", "others"]

# Frame sampling (anti-overfitting vidéo)
FRAME_STRIDE = 5   # ← clé du fix (1 frame sur 5)

# Paths
ann_train = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Train-Annotations-XML/DETRAC-Train-Annotations-XML"
ann_val   = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Test-Annotations-XML/DETRAC-Test-Annotations-XML"
img_root  = "/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Images/DETRAC-Images"


out_images_train = "data/images/train"
out_labels_train = "data/labels/train"
out_images_val   = "data/images/val"
out_labels_val   = "data/labels/val"

for d in [
    out_images_train, out_labels_train,
    out_images_val, out_labels_val
]:
    os.makedirs(d, exist_ok=True)

# ==================================================
# CLASS MAPPING
# ==================================================

def map_vehicle_type(vtype):
    """
    car  -> 0
    van  -> 0
    bus / others -> ignored
    """
    if vtype in KEEP_TYPES:
        return 0
    return None

# ==================================================
# CONVERSION BOX
# ==================================================

def convert_to_yolo(left, top, width, height, img_w, img_h):
    xmin = float(left)
    ymin = float(top)
    xmax = xmin + float(width)
    ymax = ymin + float(height)

    return (
        ((xmin + xmax) / 2) / img_w,
        ((ymin + ymax) / 2) / img_h,
        (xmax - xmin) / img_w,
        (ymax - ymin) / img_h
    )

# ==================================================
# XML PROCESSING
# ==================================================

def process_xml(xml_path, mode="train"):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    seq_name = root.attrib["name"]
    img_dir = os.path.join(img_root, seq_name)

    out_img_dir = out_images_train if mode == "train" else out_images_val
    out_lbl_dir = out_labels_train if mode == "train" else out_labels_val

    for frame in root.findall("frame"):
        frame_num = int(frame.attrib["num"])

        # FRAME SAMPLING (clé anti-overfitting)
        if frame_num % FRAME_STRIDE != 0:
            continue

        img_name = f"img{frame_num:05d}.jpg"
        img_path = os.path.join(img_dir, img_name)

        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        target_list = frame.find("target_list")
        label_lines = []

        if target_list is not None:
            for obj in target_list.findall("target"):
                attr = obj.find("attribute")
                if attr is None:
                    continue

                vtype = attr.attrib.get("vehicle_type")
                cls_id = map_vehicle_type(vtype)

                if cls_id is None:
                    continue

                box = obj.find("box")
                if box is None:
                    continue

                x, y, bw, bh = convert_to_yolo(
                    box.attrib["left"],
                    box.attrib["top"],
                    box.attrib["width"],
                    box.attrib["height"],
                    w, h
                )

                label_lines.append(
                    f"{cls_id} {x:.6f} {y:.6f} {bw:.6f} {bh:.6f}"
                )

        out_name = f"{seq_name}_img{frame_num:05d}"

        # écrire label même vide (image négative)
        with open(os.path.join(out_lbl_dir, out_name + ".txt"), "w") as f:
            if label_lines:
                f.write("\n".join(label_lines))

        dst_img = os.path.join(out_img_dir, out_name + ".jpg")
        if not os.path.exists(dst_img):
            shutil.copy(img_path, dst_img)

# ==================================================
# EXECUTION/kaggle/input/datasets/bratjay/ua-detrac-orig/DETRAC-Images
# ==================================================

print("Conversion TRAIN (frame sampling activé)...")
for xml_file in glob(os.path.join(ann_train, "*.xml")):
    process_xml(xml_file, mode="train")

print("Conversion VAL (frame sampling activé)...")
for xml_file in glob(os.path.join(ann_val, "*.xml")):
    process_xml(xml_file, mode="val")

print("Dataset YOLO UA-DETRAC (car + van) prêt — frame redundancy réduite.")

Conversion TRAIN (frame sampling activé)...
Conversion VAL (frame sampling activé)...
✅ Dataset YOLO UA-DETRAC (car + van) prêt — frame redundancy réduite.


# Model yolo11s 

In [2]:
!pip -q install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.0 MB/s eta 0:00:00a 0:00:01


In [8]:
from ultralytics import YOLO
import os, yaml

class YOLOTrainer:
    def __init__(
        self,
        model_name="yolo11n.pt",
        data_yaml="dataset.yaml",
        imgsz=960,
        device=0,
        project="runs_det",
        run_name="yolo12s_detrac_carvan"
    ):
        """
        YOLO trainer adapté au dataset UA-DETRAC
        (car + van fusionnés → classe unique 'car').
        """
        self.model = YOLO(model_name)
        self.data_yaml = data_yaml
        self.imgsz = imgsz
        self.device = device
        self.project = project
        self.run_name = run_name

    def train(
        self,
        epochs=30,
        batch=16,
        workers=4,

        # fine-tuning
        freeze=12,
        patience=4,
        lr0=0.005,
        lrf=0.01,

        # loss weights
        box=7.5,
        cls=0.5,

    ):
        """
        Entraînement YOLO optimisé pour scènes fixes
        et détection de véhicules légers.
        """

        results = self.model.train(
            data=self.data_yaml,
            imgsz=self.imgsz,
            epochs=epochs,
            batch=batch,
            device=self.device,
            workers=workers,

            # stabilité apprentissage
            freeze=freeze,
            patience=patience,
            lr0=lr0,
            lrf=lrf,

            # pondération pertes
            box=box,
            cls=cls,
            
            # NMS (impact direct precision / recall)
            conf=0.30,          # seuil optimal (vu sur F1 curve)
            iou=0.60,           # NMS plus permissif (↑ recall)
            max_det=300,        # scènes denses
            agnostic_nms=True,  # 1 seule classe (TRÈS important)
           
            project=self.project,
            name=self.run_name,
        )

        return results

    def validate(self):
        """
        Évaluation finale sur validation UA-DETRAC.
        """
        return self.model.val(
            data=self.data_yaml,
            imgsz=self.imgsz,
            #device=self.device
        )

    def export_onnx(self):
        """
        Export ONNX pour edge deployment / quantization.
        """
        return self.model.export(
            format="onnx",
            imgsz=self.imgsz,
            simplify=True
        )

data = {
    "train": "data/images/train",
    "val": "data/images/val",
    "nc": 1,
    "names":  ["car"]
}

with open("dataset.yaml", "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("dataset.yaml créé :", "dataset.yaml")

trainer = YOLOTrainer(
    model_name="yolo11s.pt",
    data_yaml="dataset.yaml",
    imgsz=768,
    project="runs",
    run_name="yolov8n_detrac_car_rot"
)

In [10]:
data = {
    "train": "data/images/train",
    "val": "data/images/val",
    "nc": 1,
    "names":  ["car"]
}

with open("dataset.yaml", "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("dataset.yaml créé :", "dataset.yaml")

trainer = YOLOTrainer(
    model_name="yolo11s.pt",
    data_yaml="dataset.yaml",
    imgsz=768,
    project="runs",
    run_name="yolov8n_detrac_car_rot"
)

trainer.train()

dataset.yaml créé : dataset.yaml
Ultralytics 8.4.46 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=True, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.3, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=12, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.6, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_detrac_car_rot-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ove

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7aac08ddeae0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 